# Emotion Detection

Given an image of a facial expression, we want to determine what emotion is being portrayed. This is a multiclass classification problem as there are a total of 7 different emotions that we can classify instances as: angry, disgust, fear, happy, sad, surprised, and neutral. Our goal is to build and compare different Convolutional Neural Network (CNN) models to classify these emotions from inputted images.

This notebook contains all exploratory experiments, model iterations, and evaluation runs used to develop the final model.

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import pickle

2025-11-12 18:14:04.492479: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Load Preprocessed Data

In [3]:
data = np.load('../data/processed_data.npz')

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

X_train_normalized = data['X_train_normalized']
X_test_normalized = data['X_test_normalized']
X_val_normalized = data['X_val_normalized']

y_train = data['y_train']
y_test = data['y_test']
y_val = data['y_val']

y_train_cat = data['y_train_cat']
y_test_cat = data['y_test_cat']
y_val_cat = data['y_val_cat']

## Utilities

### Data Augmentation

There is a significant difference in the number of images across the different emotion classes. This could potentially lead to overfitting as our model will be trained using more images from certain classes than others. In order to get more data to use to train our model and to fix the imbalance, we will oversample the training data by performing image augmentation. We choose to oversample instead of undersample as undersampling can lead to the loss of important data. We also do not want to duplicate images as this can also lead to overfitting.

Perform data augmentation on all classes (not just minority ones) to create new versions of existing images. This teaches the model to handle small variations in images.

In [ ]:
# layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

Oversample minority classes.

In [ ]:
# find all unique class labels and count how many samples belong to each class
classes, counts = np.unique(y_train, return_counts=True)

# calculate how often to sample each class, minority classes get higher weights
total = np.sum(counts)
sample_weights = total / (len(classes) * counts)
sample_weights = sample_weights / np.sum(sample_weights) # normalize weights

# create one dataset per class
datasets = []
for c in classes:
    X_c = X_train[y_train == c]
    y_c = y_train_cat[y_train == c]
    ds_c = tf.data.Dataset.from_tensor_slices((X_c, y_c))
    ds_c = ds_c.shuffle(1000).repeat()
    datasets.append(ds_c)

augmented_ds = tf.data.Dataset.sample_from_datasets(datasets, weights=sample_weights)

def augment(X, y):
    return data_augmentation(X), y

train_ds = augmented_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

### Class Weights

Use class weights to tell the loss function to pay more attention to less represented classes (like disgust and fear).

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))

### Learning Rate Scheduler

Use a learning rate scheduler to dynamically adjust the learning rate based on validation loss. When validation loss stops improving for a set number of epochs, the scheduler will reduce the learning rate to allow the model to converge more efficiently.

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

### Helper Functions

In [ ]:
# plots accuracy and loss training curves
def plot_training_curves(history, title):
    plt.figure(figsize=(12, 5))

    # accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='train')
    plt.plot(history['val_accuracy'], label='val')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy');
    plt.legend();
    
    # loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend();

    plt.show()

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']    

In [ ]:
# evaluates model, test accuracy & loss, confusion matrix, classification report
def evaluate_model(model, X_test, y_test, title):
    # test results
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'{title} - Test Accuracy: {test_accuracy:.4f} Test Loss: {test_loss:.4f}')
    
    # confusion matrix
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
    plt.title(f'{title} - Confusion Matrix')
    plt.show()
    
    # classification report
    print(classification_report(y_true, y_pred, target_names=emotion_labels))

## Build Models


Build a Convolutional Neural Network (CNN) that classifies images by 1 of 7 emotion types.

### Model 1a (Baseline)

Start by creating a simple model with only 1 convolutional layer.

In [ ]:
model_1a = tf.keras.Sequential([
    # define input shape, 48x48 pixels, 1 channel (grayscale)
    layers.Input(shape=(48, 48, 1)),

    # convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu'),

    # convert 2D feature map into a 1D vector
    layers.Flatten(),

    # fully connected layer that learns combinations of the features detected by the convolutional layer, 64 neurons = 64 combinations/patterns
    layers.Dense(64, activation='relu'),

    # final layer with 7 neurons (one for each emotion)
    layers.Dense(7, activation='softmax')
])

model_1a.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_1a = model_1a.fit(
    X_train_normalized, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

# save history during training
with open('../history/history_1a.pkl', 'wb') as f:
    pickle.dump(history_1a.history, f)

model_1a.save('../models/model_1a.keras')

In [ ]:
plot_training_curves(history_1a, "Model 1a")
evaluate_model(model_1a, X_test_normalized, y_test_cat, "Model 1a")

Model 1a achieved 42% test accuracy, performing best on Happy (F1 = 0.61) and Surprise (F1 = 0.51) but poorly on Disgust which was not predicted correctly at all (F1 = 0.00). Model is overfitting, it performs well on training data but poorly on validation data. With only 1 convolutional layer and 1 dense layer, it is too shallow to learn and capture the complexity of facial features.

### Model 1b (Baseline + Normalization + Pooling)

Add batch normalization to help stabilize and speed up training. Add max pooling to reduce spatial dimensions to prevent overfitting and reduce computation time/parameters while still keeping the most important features. Choose a pool size of 2x2 to halve each spatial dimension at each layer.

In [ ]:
model_1b = tf.keras.Sequential([
    # define input shape, 48x48 pixels, 1 channel (grayscale)
    layers.Input(shape=(48, 48, 1)),

    # convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    # convert 2D feature map into a 1D vector
    layers.Flatten(),

    # fully connected layer that learns combinations of the features detected by the convolutional layer, 64 neurons = 64 combinations/patterns
    layers.Dense(64, activation='relu'),

    # final layer with 7 neurons (one for each emotion)
    layers.Dense(7, activation='softmax')
])

model_1b.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_1b = model_1b.fit(
    X_train_normalized, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

# save history during training
with open('../history/history_1b.pkl', 'wb') as f:
    pickle.dump(history_1b.history, f)

model_1b.save('../models/model_1b.keras')

In [ ]:
plot_training_curves(history_1b, "Model 1b")
evaluate_model(model_1b, X_test_normalized, y_test_cat, "Model 1b")

After introducing batch normalization and max pooling, Model 1b achieved a test accuracy of 43%, a slight improvement over the original baseline's 42%. While normalization stabilized training and pooling improved feature extraction, the model still displayed signs of overfitting as training accuracy rose quickly to 77% while validation accuracy plateaued around 43%. Performance remains class dependent. Model performs best on Happy (F1 = 0.57) and Surprise (F1 = 0.46). Interestingly, despite having relatively few samples, Surprise achieved good performance, likely due to its distinctive facial cues. In contrast, Disgust (F1 = 0.00) was completely missed again, likely due to it being underrepresented in the training data and its subtle hard to learn facial features. The addition of normalization and pooling helped improve generalization slightly but is not enough to handle class imbalance or complex feature extraction.

Since certain emotions have more images than others, we can introduce class weights and perform data augmentation to address the class imbalance and improve feature extraction and generalization next.

### Model 2a (Data Augmentation + Class Weights)

Introduce data augmentation and apply class weights to improve generalization and counter data imbalance by giving minority classes higher importance. Apply data augmentation to all classes.

In [ ]:
model_2a = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(7, activation='softmax')
])

model_2a.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_2a = model_2a.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history_2a.pkl', 'wb') as f:
    pickle.dump(history_2a.history, f)

model_2a.save('../models/model_2a.keras')

In [ ]:
plot_training_curves(history_2a, "Model 2a")
evaluate_model(model_2a, X_test_normalized, y_test_cat, "Model 2a")

Expected generalization to improve after applying data augmentation and class weights to the model, but performance actually decreased slightly from 43% to 38% test accuracy. This suggests that while the model is now exposed to more variation and data diversity, it still lacks the depth to learn complex facial features (underfitting). However, even though overall accuracy decreased, the performance is now more balanced across classes. Notably, Disgust which was never predicted correctly at all in Model 1, is now being recognized (albeit poorly), indicating that applying class weights helps the model pay more attention to minority classes.

### Model 2b

Oversample minority classes using class based weighted sampling to increase their representation during training. Split the training data into one dataset per emotion class and combine them with weights so that minority classes are sampled more frequently. Continue to apply data augmentation to all classes.

In [ ]:
def augment_data(X, y):
  ds = tf.data.Dataset.from_tensor_slices((X, y))
  ds = ds.shuffle(len(X)).repeat()

  def augment(X, y):
    X = tf.cast(X, tf.float32)
    X = data_augmentation(X)
    X = X / 255
    return X, y

  ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)
  return ds

classes, counts = np.unique(y_train, return_counts=True)
total = np.sum(counts)
sample_weights = total / (len(classes) * counts)
sample_weights = sample_weights / np.sum(sample_weights)


datasets = [
    augment_data(X_train[y_train == c], y_train_cat[y_train == c])
    for c in classes
]

train_ds = tf.data.Dataset.sample_from_datasets(datasets, weights=sample_weights)

In [ ]:
model_2b = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(7, activation='softmax')
])

model_2b.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_2b = model_2b.fit(
    train_ds,
    epochs=30,
    steps_per_epoch=len(X_train) // 64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

with open('../history/history_2b.pkl', 'wb') as f:
    pickle.dump(history_2b.history, f)

model_2b.save('../models/model_2b.keras')

Model memorized the oversampled training data and failed to generalize. Extreme overfitting occurred due to class duplication/repetition, resulting in very high training accuracy but extremely poor validation/test performance. Almost all predictions were biased towards minority classes. 

### Model 2c

Build on model 2b, but apply data augmentation only to minority classes while majority classes are normalized without augmentation.

In [ ]:
def augment_data(X, y):
  ds = tf.data.Dataset.from_tensor_slices((X, y))
  ds = ds.shuffle(len(X)).repeat()

  def augment(X, y):
    X = tf.cast(X, tf.float32)
    X = data_augmentation(X)
    X = X / 255
    return X, y

  ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)
  return ds

def normalize_data(X, y):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(len(X)).repeat()

    def normalize(X, y):
        X = tf.cast(X, tf.float32) / 255
        return X, y

    
    ds = ds.map(normalize, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)
    return ds

        
classes, counts = np.unique(y_train, return_counts=True)
total = np.sum(counts)

sample_weights = total / (len(classes) * counts)
sample_weights = sample_weights / np.sum(sample_weights)

# select all classes with sample count less than 50% of largest class
max_count = counts[np.argmax(counts)]
minority_classes = [c for c, count in zip(classes, counts) if count < 0.5 * max_count]
majority_classes = [c for c in classes if c not in minority_classes]

datasets = []

for c in classes:
    X_c = X_train[y_train == c]
    y_c = y_train_cat[y_train == c]

    if c in minority_classes:
        ds_c = augment_data(X_c, y_c)
    else:
        ds_c = normalize_data(X_c, y_c)

    datasets.append(ds_c)


train_ds = tf.data.Dataset.sample_from_datasets(datasets, weights=sample_weights)

Targeted augmentation of minority classes reduces extreme bias. Model shows improvement and better generalization over naive oversampling but still overfits.

Overall, model 2a performed the best. Continue to apply data augmentation + class weights without oversampling minority classes. Focus on increasing network depth to enhance feature extraction next.

### Model 3a (More Layers)

Expand model depth from a shallow single layer CNN to a deeper CNN with three convolutional layers, each followed by batch normalization and max pooling. The deeper architecture should allow the model to capture and learn more complex facial features that a shallow network cannot effectivly learn. Continue using data augmentation and class weights to improve generalization and address class imbalance.

In [ ]:
model_3a = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    # classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_3a.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_3a = model_3a.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history_3a.pkl', 'wb') as f:
    pickle.dump(history_3a.history, f)

model_3a.save('../models/model_3a.keras')

In [ ]:
plot_training_curves(history_3a, "Model 3a")
evaluate_model(model_3a, X_test, y_test_cat, "Model 3a")

By introducing additional convolutional layers, the model was able to learn and extract more complex facial features, leading to a significant improvement in performance from 38% to 52% test accuracy. Performance improved across all classes with increases in precision, recall, and F1 scores. Happy and Surprise remain the easiest emotions to identify, while minority classes such as Disgust showed notable improvement (F1 = 0.24, recall = 0.62) indicating better handling of imbalance. However, the model still struggles with confusion between similar emotions like Sad, Neutral, Fear, and Angry. Applying data augmentation and class weights helped reduced bias towards dominant classes, allowing the model to recognize minority emotions more effectively. Although mild overfitting occurred, overall the model generalizes better and demonstrates more balanced learning across classes. 

### Model 3b

Expand model depth even further by replacing single convolutional layers with full convolutional blocks, each containing two convolutional layers followed by batch normalization and max pooling. The block based architecture should increase the model's capacity to learn and extract more features. Additionally, apply dropout regularization after each block and in the dense layer to reduce overfitting and stabilize training. Increase training time from 50 epochs to 100 to allow the larger model more time to converge.

In [ ]:
model_3b = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_3b.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_3b = model_3b.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history_3b.pkl', 'wb') as f:
    pickle.dump(history_3b.history, f)

model_3b.save('../models/model_3b.keras')

The deeper architecture and introduction of dropout layers helped the model generalize more effectively and distinguish facial expressions with greater precision. Model 3b achieved a test accuracy of 56%, outperforming Model 3a's 52%. F1 scores increased across all emotions except Fear (which decreased by 0.05), with a notable increase for Sad (from 0.38 to 0.49). Happy (F1 = 0.80) and Surprise (F1 = 0.68) remain the easiest emotions to predict. Overall, Model 3b demonstrates a clear performance gain, validating that deeper CNN architectures combined with dropout regularization improve model performance and capability.


### Model 3c

Further increase model capacity by doubling the number of filters used in each convolutional block to enable network to capture even more complex features.

In [ ]:
model_3c = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_3c.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_3c = model_3c.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('..history/history_3c.pkl', 'wb') as f:
    pickle.dump(history_3c.history, f)

model_3c.save('../models/model_3c.keras')

Increasing the number of filters strengthened the model's ability to extract features leading to a stronger and more balanced performance across all emotions. The model achieved a test accuracy of 58%, a clear improvement over previous models. F1 scores improved for Angry, Disgust, and Fear, while Happy, Sad, and Surprise remained the same. Training and validation accuracies remained closely aligned with steadily decreasing loss curves indicating effective learn without overfitting. The confusion matrix shows fewer misclassifications between visually similar emotions like Sad and Neutral, demonstrating improved feature differentiation. Overall, expanding filter depth enhanced model performance.

### Model 4a (Learning Rate Scheduler)

Build on model 3c by adding a learning rate scheduler that dynamically adjusts the learning rate based on validation loss. When validation loss stops improving for a set number of epochs, the scheduler will reduce the learning rate to allow the model to converge more efficiently.

In [ ]:
model_4a = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_4a.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_4a = model_4a.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

with open('..history/history_4a.pkl', 'wb') as f:
    pickle.dump(history_4a.history, f)

model_4a.save('../models/model_4a.keras')

In [ ]:
plot_training_curves(history_4a, "Model 4a")
evaluate_model(model_4a, X_test, y_test_cat, "Model 4a")

Adding the learning rate scheduler resulted in more stable training and smoother convergence resulting in a test accuracy of 58%. Training and validation curves remain closely aligned, indicating no overfitting. F1 scores remained similar to those of model 3c with improvements for Happy, Surprise, and Neutral while Disgust and Fear saw slight declines. Overall, model maintained the balanced generalization of Model 3c while achieving slightly improved performance and smoother optimization.

### Model 4b (Global Average Pooling)

Build on Model 4a and replace the Flatten layer with a GlobalAveragePooling2D (GAP) layer. While the Flatten layer preserves all spatial information, it significantly increases the number of parameters which makes the model more prone to overfitting. On the other hand, the GAP layer compresses each feature map into a single value, reducing the total number of parameters and overall model complexity, which helps prevent overfitting and improves efficiency while still preserving important feature information.

In [ ]:
model_4b = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_4b.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_4b = model_4b.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

with open('..history/history_4b.pkl', 'wb') as f:
    pickle.dump(history_4b.history, f)

model_4b.save('../models/model_4b.keras')

In [ ]:
plot_training_curves(history_4b, "Model 4b")
evaluate_model(model_4b, X_test, y_test_cat, "Model 4b")

Replacing the Flatten layer with a GAP layer led to improved generalization and a higher test accuracy of 59%. Training and validation curves remained stable with minimal overfitting.  F1 scores across all emotions either improved or remained the same, with notable improvement for Disgust (0.36 to 0.41) and Fear (0.31 to 0.36). The confusion matrix also shows reduced misclassifications between visually similar emotions (Sad and Neutral). Overall, the model achieved higher performance with reduced model complexity.


### Model 4c (Regularization Techniques)

Add label smoothing and L2 regularization as complementary regularization techniques. Label smoothing reduces overconfidence in the model's predictions by distributing a portion of probability across all classes instead of predicting one emotion with complete certainty. This allows minority and harder to predict classes to receive more balanced consideration. L2 regularization penalizes large weights across all layers, encouraging smaller weights to improve generalization, reduce overfitting, and stabilize training.

In [ ]:
l2_strength = 1e-3

model_4c = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_4c.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_4c = model_4c.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

with open('..history/history_4c.pkl', 'wb') as f:
    pickle.dump(history_4c.history, f)

model_4d.save('../models/model_4c.keras')

In [ ]:
plot_training_curves(history_4c, "Model 4c")
evaluate_model(model_4c, X_test, y_test_cat, "Model 4c")

Model achieved a test accuracy of 62% and a test loss of 1.234, slightly higher than Model 4b, but demonstrated improved performance across most emotions. Disgust (F1 = 0.46) and Fear (F1 = 0.40) showed the most improvement, while Sad saw a minimal decline. The confusion matrix also shows fewer misclassifications between visually similar emotions, suggesting better feature discrimination. Overall, Model 4c demonstrates stronger generalization and more balanced performance across all classes, benefiting from the effects of both label smoothing and L2 regularization.

### Model 5a

Try adding 1 more convolutional block and reduce initial filters to 32 to help model start training more stably.

In [ ]:
l2_strength = 1e-3

model_5a = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_strength)),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model_5a.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_5a = model_5a.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr]
)

with open('..history/history_5a.pkl', 'wb') as f:
    pickle.dump(history_5a.history, f)

model_5a.save('../models/model_5a.keras')

Model achieved a test accuracy of 63%, the highest among all versions so far. Training and validation curves show a small gap, indicating minimal overfitting. The model converged steadily over 100 epochs. F1 scores improved or remained stable across all classes. Happy, Surprise, and Neutral performed best while Disgust and Fear continue to underperform. Overall, this model shows consistent improvement across most metrics, a small gain over the previous model.